This is a notebook to visualise the spectroscopy data from the USGS data set

In [33]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
import glob


In [59]:
aster_speclib = USGSSatelliteSpectra('ASCIIdata')

Found data directory: ASCIIdata\ASCIIdata_splib07b_rsASTER
Loading normal wavelength data from: ASCIIdata\ASCIIdata_splib07b_rsASTER\S07ASTER_Wavelengths_ASTER_(9_bands)_microns.txt
Loaded normal wavelength data: 9 bands
Loading hyperspectral wavelength data from: ASCIIdata\ASCIIdata_splib07b_rsASTER\S07ASTER_Wavelengths_in_microns_for_ASTER_SRFs.txt
Loaded hyperspectral wavelength data: 385 channels
Loading bandpass data (nm) from: ASCIIdata\ASCIIdata_splib07b_rsASTER\S07ASTER_Bandpass_(FWHM)_ASTER_(9_bands)_nm.txt
Loaded bandpass data (nm): 9 values
Loading bandpass data (microns) from: ASCIIdata\ASCIIdata_splib07b_rsASTER\S07ASTER_Bandpass_(FWHM)_ASTER_(9_bands)_um.txt
Loaded bandpass data (microns): 9 values
Found 9 band response function files
Loaded Band 1 (Band_1): 385 values
Loaded Band 2 (Band_2): 385 values
Loaded Band 3N (Band_3N): 385 values
Loaded Band 4 (Band_4): 385 values
Loaded Band 5 (Band_5): 385 values
Loaded Band 6 (Band_6): 385 values
Loaded Band 7 (Band_7): 385 v

In [ ]:
aster_minerals = aster_speclib.load_minerals()

In [60]:
aster_band_info = aster_speclib.get_band_info()

In [58]:

class USGSSatelliteSpectra:
    def __init__(self, base_dir, satellite='ASTER'):
        """
        Initialize the USGS Spectral Library satellite data loader
        
        Parameters:
        -----------
        base_dir : str
            Base directory containing USGS Spectral Library files
        satellite : str
            Satellite sensor name (e.g., 'ASTER', 'LSAT8', 'SNTL2', 'WV3')
        """
        self.base_dir = Path(base_dir)
        self.satellite = satellite
        self.prefix = f"S07{satellite}_"
        
        # Find the appropriate data directory
        self.data_dir = list(self.base_dir.glob(f"**/ASCIIdata_splib07b_rs{satellite}"))[0]
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Could not find data directory for {satellite}")
        
        print(f"Found data directory: {self.data_dir}")
        
        # Initialise collections
        self.spectra = {}
        self.wavelengths = None           # Normal wavelengths (for plotting with spectra)
        self.wavelengths_hp = None        # Hyperspectral wavelengths (for band data)
        self.bandpass_nm = None           # Bandpass in nanometers
        self.bandpass_micron = None       # Bandpass in microns
        self.bands = {}                   # Dictionary to store individual band response functions
        
        # Load wavelength data
        self._load_wavelength_data()
        self._load_band_data()
    
    def _load_wavelength_data(self):
        """Load wavelength and bandpass data for the satellite sensor"""
        
        # Load normal wavelengths (with "bands" in filename)
        wavelength_pattern = f"{self.prefix}Wavelengths*bands*.txt"
        wavelength_files = list(self.data_dir.glob(wavelength_pattern))
        
        if wavelength_files:
            wavelength_file = wavelength_files[0]
            print(f"Loading normal wavelength data from: {wavelength_file}")
            self.wavelengths = np.loadtxt(wavelength_file, skiprows=1)
            print(f"Loaded normal wavelength data: {len(self.wavelengths)} bands")
        
        # Load hyperspectral wavelengths ()
        hp_wavelength_pattern = f"{self.prefix}Wavelengths_*S*R*F*.txt"
        hp_wavelength_files = list(self.data_dir.glob(hp_wavelength_pattern))
        
        if hp_wavelength_files:
            hp_wavelength_file = hp_wavelength_files[0]
            print(f"Loading hyperspectral wavelength data from: {hp_wavelength_file}")
            self.wavelengths_hp = np.loadtxt(hp_wavelength_file, skiprows=1)
            print(f"Loaded hyperspectral wavelength data: {len(self.wavelengths_hp)} channels")
        
        # Load bandpass data in nanometers
        bandpass_nm_pattern = f"{self.prefix}Bandpass*nm.txt"
        bandpass_nm_files = list(self.data_dir.glob(bandpass_nm_pattern))
        
        if bandpass_nm_files:
            bandpass_nm_file = bandpass_nm_files[0]
            print(f"Loading bandpass data (nm) from: {bandpass_nm_file}")
            self.bandpass_nm = np.loadtxt(bandpass_nm_file, skiprows=1)
            print(f"Loaded bandpass data (nm): {len(self.bandpass_nm)} values")
        
        # Load bandpass data in microns
        bandpass_micron_pattern = f"{self.prefix}Bandpass*um.txt"
        bandpass_micron_files = list(self.data_dir.glob(bandpass_micron_pattern))
        
        if bandpass_micron_files:
            bandpass_micron_file = bandpass_micron_files[0]
            print(f"Loading bandpass data (microns) from: {bandpass_micron_file}")
            self.bandpass_micron = np.loadtxt(bandpass_micron_file, skiprows=1)
            print(f"Loaded bandpass data (microns): {len(self.bandpass_micron)} values")
        
    def _load_band_data(self):
        """Load individual band response functions"""
        
        # Pattern to match band files: S07LSAT8_SRF_Band_3_Landsat8_Green.txt
        band_pattern = f"{self.prefix}*SRF*Band*.txt"
        band_files = list(self.data_dir.glob(band_pattern))
        
        print(f"Found {len(band_files)} band response function files")
        
        for band_file in band_files:
            try:
                # Parse the band information from filename
                filename = band_file.name
                # Extract band number and name
               
                parts = filename.replace(self.prefix, '').replace('.txt', '').split('_')
                
                if len(parts) >= 4 and parts[0] == 'SRF' and 'Band' in parts:
                    # Extract band number
                    band_number = parts[parts.index('Band')+1]
                    # Extract band name (everything after the satellite name)
                    band_name_parts = parts[4:]  # Skip 'SRF', 'Band', number, and satellite name
                    band_name = '_'.join(band_name_parts) if band_name_parts else f"Band_{band_number}"
                    
                    # Load the band response function
                    band_data = np.loadtxt(band_file, skiprows=1)
                    
                    # Store band information
                    self.bands[band_number] = {
                        'band_name': band_name,
                        'response': band_data,
                        'filename': filename,
                        'wavelengths': self.wavelengths_hp if self.wavelengths_hp is not None else np.arange(len(band_data))
                    }
                    
                    print(f"Loaded Band {band_number} ({band_name}): {len(band_data)} values")
                    
            except Exception as e:
                print(f"Error loading band file {band_file}: {str(e)}")
        
        print(f"Successfully loaded {len(self.bands)} band response functions")
        
    
    def _parse_filename(self, filename):
        """
        Parse the spectrum filename to extract metadata
        
        Example: S07ASTER_Actinolite_HS22.4B_ASDFRb_AREF.txt
        """
        basename = os.path.basename(filename)
        # Remove prefix and file extension
        parts = basename.replace(self.prefix, '').replace('.txt', '').split('_')
        
        if len(parts) < 3:
            return None
        
        # Extract material, sample ID, and measurement info
        material = parts[0]
        
        # Extract spectrometer and purity code
        spec_code = parts[-2]
        spec_match = re.match(r'([A-Z]+[0-9]*)([a-z]+)', spec_code)
        
        if not spec_match:
            return None
            
        spectrometer = spec_match.group(1)
        purity = spec_match.group(2)
        
        # Extract sample ID (everything between material and spectrometer)
        sample_id = '_'.join(parts[1:-2])
        
        # Extract measurement type
        measurement_type = parts[-1]
        
        return {
            'material': material,
            'sample_id': sample_id,
            'spectrometer': spectrometer,
            'purity': purity,
            'measurement_type': measurement_type,
            'filename': basename,
            'full_path': filename,
            'satellite': self.satellite
        }

    def load_minerals(self, max_samples=None):
        """
        Load mineral spectra from Chapter M
        
        Parameters:
        -----------
        max_samples : int or None
            Maximum number of samples to load (None for all)
            
        Returns:
        --------
        dict
            Dictionary of loaded mineral spectra
        """
        # Find the minerals directory
        minerals_dir = self.data_dir / "ChapterM_Minerals"
        if not minerals_dir.exists():
            raise FileNotFoundError(f"Could not find minerals directory: {minerals_dir}")
        
        # Find all spectrum files
        spectrum_files = list(minerals_dir.glob(f"{self.prefix}*.txt"))
        
        if max_samples and max_samples < len(spectrum_files):
            spectrum_files = spectrum_files[:max_samples]
        
        print(f"Found {len(spectrum_files)} mineral spectra files")
        
        # Load each spectrum
        for i, filename in enumerate(spectrum_files):
            if i % 100 == 0:
                print(f"Loading spectrum {i+1}/{len(spectrum_files)}")
            
            try:
                # Parse filename for metadata
                metadata = self._parse_filename(str(filename))
                if not metadata:
                    print(f"Could not parse filename: {filename}")
                    continue
                
                # Load spectrum data
                spectrum = np.loadtxt(filename, skiprows=1)
                
                # Replace deleted channels with NaN
                spectrum[spectrum < -1e30] = np.nan
                
                # Create a unique key
                key = f"{metadata['material']}_{metadata['sample_id']}"
                
                # Store the data
                self.spectra[key] = {
                    'metadata': metadata,
                    'spectrum': spectrum
                }
                
            except Exception as e:
                print(f"Error loading spectrum {filename}: {str(e)}")
        
        print(f"Successfully loaded {len(self.spectra)} mineral spectra")
        return self.spectra
    
    def get_band_info(self):
        """
        Get summary information about loaded bands
        
        Returns:
        --------
        pandas.DataFrame
            DataFrame with band information
        """
        if not self.bands:
            print("No band data loaded")
            return None
        
        band_info = []
        
        for band_num, band_data in sorted(self.bands.items()):
            response = band_data['response']
            wavelengths_band = band_data['wavelengths']
            
            # Calculate band statistics
            min_len = min(len(wavelengths_band), len(response))
            wavelengths_band = wavelengths_band[:min_len]
            response = response[:min_len]
            
            peak_idx = np.argmax(response)
            peak_wavelength = wavelengths_band[peak_idx]
            peak_response = response[peak_idx]
            
            # Calculate effective wavelength (weighted average)
            total_response = np.sum(response)
            if total_response > 0:
                eff_wavelength = np.sum(wavelengths_band * response) / total_response
            else:
                eff_wavelength = peak_wavelength
            
            # Calculate FWHM
            half_max = peak_response / 2
            indices = np.where(response >= half_max)[0]
            if len(indices) > 0:
                fwhm_start = wavelengths_band[indices[0]]
                fwhm_end = wavelengths_band[indices[-1]]
                fwhm = fwhm_end - fwhm_start
            else:
                fwhm = 0
            
            band_info.append({
                'Band_Number': band_num,
                'Band_Name': band_data['band_name'],
                'Peak_Wavelength_um': peak_wavelength,
                'Effective_Wavelength_um': eff_wavelength,
                'FWHM_um': fwhm,
                'Peak_Response': peak_response,
                'Min_Wavelength_um': np.min(wavelengths_band),
                'Max_Wavelength_um': np.max(wavelengths_band)
            })

        return pd.DataFrame(band_info)
    
    def plot_band_responses_detailed(self, figsize=(15, 10)):
        """
        Plot detailed band response functions for the satellite
        
        Parameters:
        -----------
        figsize : tuple
            Figure size
            
        Returns:
        --------
        matplotlib.figure.Figure
            The figure object
        """
        if not self.bands or self.wavelengths_hp is None:
            print("No band response data available")
            return None
        
        # Sort bands by band number
        sorted_bands = sorted(self.bands.items())
        num_bands = len(sorted_bands)
        
        # Create subplots
        cols = min(3, num_bands)
        rows = (num_bands + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=figsize)
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1 or cols == 1:
            axes = axes.flatten()
        else:
            axes = axes.flatten()
        
        # Plot each band
        for i, (band_num, band_info) in enumerate(sorted_bands):
            if i >= len(axes):
                break
                
            ax = axes[i]
            response = band_info['response']
            wavelengths_band = band_info['wavelengths']
            
            # Ensure wavelengths and response have same length
            min_len = min(len(wavelengths_band), len(response))
            wavelengths_band = wavelengths_band[:min_len]
            response = response[:min_len]
            
            # Plot the response function
            ax.fill_between(wavelengths_band, 0, response, alpha=0.6, color=f'C{i}')
            ax.plot(wavelengths_band, response, color=f'C{i}', linewidth=2)
            
            # Calculate and display band statistics
            peak_idx = np.argmax(response)
            peak_wavelength = wavelengths_band[peak_idx]
            peak_response = response[peak_idx]
            
            # Calculate FWHM
            half_max = peak_response / 2
            indices = np.where(response >= half_max)[0]
            if len(indices) > 0:
                fwhm_start = wavelengths_band[indices[0]]
                fwhm_end = wavelengths_band[indices[-1]]
                fwhm = fwhm_end - fwhm_start
            else:
                fwhm = 0
            
            ax.set_title(f"Band {band_num}: {band_info['band_name']}\n"
                        f"Peak: {peak_wavelength:.3f} μm\n"
                        f"FWHM: {fwhm:.3f} μm")
            ax.set_xlabel('Wavelength (μm)')
            ax.set_ylabel('Response')
            ax.grid(True, linestyle='--', alpha=0.3)
            
            # Add vertical line at peak
            ax.axvline(x=peak_wavelength, color='red', linestyle='--', alpha=0.8)
            
            # Add FWHM markers
            if fwhm > 0:
                ax.axvspan(fwhm_start, fwhm_end, alpha=0.2, color='gray')
        
        # Hide unused subplots
        for i in range(num_bands, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.suptitle(f'{self.satellite} Band Response Functions Detail', fontsize=16, y=1.02)
        
        return fig

